In [2]:
import nilearn
import re
import nibabel as nib
import pandas as pd
import numpy as np
from nilearn.datasets import fetch_adhd
import matplotlib as plt
from nilearn.maskers import NiftiMasker
import os
from nilearn.image import clean_img

adhd = fetch_adhd(n_subjects=5)  # all subjects; large download
pheno = pd.DataFrame(adhd.phenotypic)

# Check the column names before using them
print("Phenotypic columns:", pheno.columns.tolist())
print("Sample size:", len(pheno))


[fetch_adhd] Dataset directory found: C:\Users\User\nilearn_data\adhd

Phenotypic columns: ['Unnamed: 0', 'Subject', 'Rest.Scan', 'MeanFD', 'NumFD_greater_than_0.20', 'rootMeanSquareFD', 'FDquartile.top1.4thFD.', 'PercentFD_greater_than_0.20', 'MeanDVARS', 'MeanFD_Jenkinson', 'site', 'sibling_id', 'data_set', 'age', 'sex', 'handedness', 'full_2_iq', 'full_4_iq', 'viq', 'piq', 'iq_measure', 'tdc', 'adhd', 'adhd_inattentive', 'adhd_combined', 'adhd_subthreshold', 'diagnosis_using_cdis', 'notes', 'sess_1_anat_2', 'oppositional', 'cog_inatt', 'hyperac', 'anxious_shy', 'perfectionism', 'social_problems', 'psychosomatic', 'conn_adhd', 'restless_impulsive', 'emot_lability', 'conn_gi_tot', 'dsm_iv_inatt', 'dsm_iv_h_i', 'dsm_iv_tot', 'study', 'sess_1_rest_1', 'sess_1_rest_1_eyes', 'sess_1_rest_2', 'sess_1_rest_2_eyes', 'sess_1_rest_3', 'sess_1_rest_3_eyes', 'sess_1_rest_4', 'sess_1_rest_4_eyes', 'sess_1_rest_5', 'sess_1_rest_5_eyes', 'sess_1_rest_6', 'sess_1_rest_6_eyes', 'sess_1_anat_1', 'sess_1_which_anat', 'sess_2_rest_1', 'sess_2_rest_1_eyes', 'sess_2_rest_2',

In [5]:
TRANS_COLS = ["motion-x", "motion-y", "motion-z"]
ROT_COLS = ["motion-pitch", "motion-roll", "motion-yaw"]
MOTION_COLS = [
    "motion-pitch",
    "motion-roll",
    "motion-yaw",
    "motion-x",
    "motion-y",
    "motion-z"
]
def friston24(conf):
    """6 motion parameters + their derivatives + squares of both = 24 regressors."""
    m = conf[MOTION_COLS].to_numpy(dtype=float)
    deriv = np.vstack([np.zeros((1, m.shape[1])), np.diff(m, axis=0)])
    base = np.hstack([m, deriv])
    return np.hstack([base, base ** 2])



nuisance_all = []

def build_nuisance(conf):

    # Friston-24
    motion = friston24(conf)

    # WM and CSF
    tissue = conf[["wm", "csf"]].to_numpy(dtype=float)

    # Combine all regressors
    nuisance = np.hstack([motion, tissue])

    return nuisance



def framewise_displacement(conf, radius=50):
    """Power FD. Rotations are in degrees and are converted to radians."""
    trans = np.abs(np.diff(conf[TRANS_COLS].to_numpy(float), axis=0)).sum(axis=1)
    rot_rad = np.deg2rad(conf[ROT_COLS].to_numpy(float))
    rot = np.abs(np.diff(rot_rad, axis=0)).sum(axis=1)
    return np.concatenate([[0.0], trans + radius * rot])

FD_THRESHOLD = 0.5  

def scrub_mask(conf, threshold=FD_THRESHOLD):
    """True = keep volume, False = censor volume."""
    fd = framewise_displacement(conf)
    return fd <= threshold, fd



FD_THRESHOLD = 0.5
TR = 2.0
OUT_DIR = "cleaned_bold"
os.makedirs(OUT_DIR, exist_ok=True)

qc_rows = []

for func_file, conf_file in zip(adhd.func, adhd.confounds):
    sid = re.search(r"(\d+)", func_file).group(1)
    print(f"Processing {sid}...")

    try:
        conf = pd.read_csv(conf_file, sep="\t")
        n_vols = nib.load(func_file).shape[-1]
        assert len(conf) == n_vols, "Confound rows and image volumes differ"

        # 1. FD and scrubbing: indices of volumes to keep
        fd = framewise_displacement(conf)
        keep = np.where(fd <= FD_THRESHOLD)[0]

        # 2. Nuisance regressors: Friston-24 + WM + CSF, full length
        nuisance = build_nuisance(conf)
        assert nuisance.shape == (n_vols, 26)

        # 3. Regression, filtering, and censoring
        cleaned = clean_img(
            func_file,
            confounds=nuisance,
            t_r=TR,
            detrend=True,
            high_pass=0.01,
            low_pass=0.1,
            standardize="zscore_sample",
            clean__sample_mask=keep,
        )
        assert cleaned.shape[-1] == len(keep), "Output length differs from retained volumes"

        out_path = os.path.join(OUT_DIR, f"sub-{sid}_cleaned.nii.gz")
        cleaned.to_filename(out_path)

        qc_rows.append({
            "sid": sid,
            "n_volumes": n_vols,
            "n_retained": len(keep),
            "pct_censored": 100 * (1 - len(keep) / n_vols),
            "mean_fd": float(fd[1:].mean()),
            "output": out_path,
        })

    except Exception as e:
        qc_rows.append({"sid": sid, "error": str(e)})
        print(f"Subject {sid} failed: {e}")


Processing 0010042...
Processing 0010064...
Processing 0010128...
Processing 0021019...
Processing 0023008...
